In [1]:
import polars as pl

In [17]:
high_confidence = catalog.load("landing.onsides.high_confidence")
vocab_rxnorm_ingredient = catalog.load("landing.onsides.vocab_rxnorm_ingredient")
vocab_meddra_adverse_effect = catalog.load(
    "landing.onsides.vocab_meddra_adverse_effect"
)

[07/02/25 18:49:41] INFO     Loading data from landing.onsides.high_confidence            ]8;id=981822;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=633200;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (ZipDataset)...                                                                       

                    INFO     Loading data from landing.onsides.vocab_rxnorm_ingredient    ]8;id=57864;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=972942;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (ZipDataset)...                                                                       

                    INFO     Loading data from                                            ]8;id=294722;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=135850;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             landing.onsides.vocab_meddra_adverse_effect (ZipDataset)...                           

In [18]:
high_confidence = high_confidence.with_columns(
    [
        pl.col("ingredient_id").map_elements(
            lambda x: f"RXNORM:{x}", return_dtype=pl.Utf8
        ),
        pl.col("effect_meddra_id").map_elements(
            lambda x: f"meddra:{x}", return_dtype=pl.Utf8
        ),
    ]
)
high_confidence

ingredient_id,effect_meddra_id
str,str
"""RXNORM:68149""","""meddra:10037844"""
"""RXNORM:6851""","""meddra:10024382"""
"""RXNORM:10395""","""meddra:10019211"""
"""RXNORM:1100072""","""meddra:10017076"""
"""RXNORM:5487""","""meddra:10008479"""
…,…
"""RXNORM:596724""","""meddra:10013946"""
"""RXNORM:342369""","""meddra:10028836"""
"""RXNORM:56946""","""meddra:10029202"""


In [19]:
vocab_rxnorm_ingredient = vocab_rxnorm_ingredient.filter(
    pl.col("rxnorm_id").str.contains(r"^\d+$")
).with_columns(
    [
        pl.col("rxnorm_id").map_elements(lambda x: f"RXNORM:{x}", return_dtype=pl.Utf8),
    ]
)
vocab_rxnorm_ingredient

rxnorm_id,rxnorm_name,rxnorm_term_type
str,str,str
"""RXNORM:1006297""","""Aspergillus niger var. niger a…","""Ingredient"""
"""RXNORM:10814""","""liothyronine""","""Ingredient"""
"""RXNORM:10829""","""trimethoprim""","""Ingredient"""
"""RXNORM:108542""","""Meningitis vaccine""","""Ingredient"""
"""RXNORM:10869""","""tropicamide""","""Ingredient"""
…,…,…
"""RXNORM:3418""","""dihydroergotamine""","""Ingredient"""
"""RXNORM:34316""","""potassium nitrate""","""Ingredient"""
"""RXNORM:1303851""","""Influenza A virus vaccine, A-C…","""Ingredient"""


In [20]:
vocab_meddra_adverse_effect = vocab_meddra_adverse_effect.with_columns(
    [
        pl.col("meddra_id").map_elements(lambda x: f"meddra:{x}", return_dtype=pl.Utf8),
    ]
)
vocab_meddra_adverse_effect

meddra_id,meddra_name,meddra_term_type
str,str,str
"""meddra:10000021""","""21-hydroxylase deficiency""","""PT"""
"""meddra:10000045""","""Abdomen enlarged""","""LLT"""
"""meddra:10000050""","""Abdominal adhesions""","""PT"""
"""meddra:10000054""","""Abdominal aortic aneurysm""","""LLT"""
"""meddra:10000055""","""Abdominal colic""","""LLT"""
…,…,…
"""meddra:10090314""","""Potential for medication error""","""LLT"""
"""meddra:10090402""","""Chest wall rigidity""","""PT"""
"""meddra:10090404""","""Diffuse alveolar haemorrhage""","""LLT"""


In [22]:
j1 = high_confidence.join(
    vocab_rxnorm_ingredient, left_on="ingredient_id", right_on="rxnorm_id", how="inner"
)
j1

ingredient_id,effect_meddra_id,rxnorm_name,rxnorm_term_type
str,str,str,str
"""RXNORM:1100072""","""meddra:10017076""","""abiraterone""","""Ingredient"""
"""RXNORM:1100072""","""meddra:10061024""","""abiraterone""","""Ingredient"""
"""RXNORM:1100072""","""meddra:10020772""","""abiraterone""","""Ingredient"""
"""RXNORM:1100072""","""meddra:10021789""","""abiraterone""","""Ingredient"""
"""RXNORM:1100072""","""meddra:10007554""","""abiraterone""","""Ingredient"""
…,…,…,…
"""RXNORM:300195""","""meddra:10037844""","""tenofovir disoproxil""","""Ingredient"""
"""RXNORM:300195""","""meddra:10023673""","""tenofovir disoproxil""","""Ingredient"""
"""RXNORM:300195""","""meddra:10047890""","""tenofovir disoproxil""","""Ingredient"""


In [23]:
j2 = j1.join(
    vocab_meddra_adverse_effect,
    left_on="effect_meddra_id",
    right_on="meddra_id",
    how="inner",
)
j2

ingredient_id,effect_meddra_id,rxnorm_name,rxnorm_term_type,meddra_name,meddra_term_type
str,str,str,str,str,str
"""RXNORM:358258""","""meddra:10000060""","""bortezomib""","""Ingredient""","""Abdominal distension""","""PT"""
"""RXNORM:73494""","""meddra:10000081""","""telmisartan""","""Ingredient""","""Abdominal pain""","""PT"""
"""RXNORM:358258""","""meddra:10000081""","""bortezomib""","""Ingredient""","""Abdominal pain""","""PT"""
"""RXNORM:68149""","""meddra:10000081""","""mycophenolate mofetil""","""Ingredient""","""Abdominal pain""","""PT"""
"""RXNORM:17767""","""meddra:10000081""","""amlodipine""","""Ingredient""","""Abdominal pain""","""PT"""
…,…,…,…,…,…
"""RXNORM:5487""","""meddra:10061461""","""hydrochlorothiazide""","""Ingredient""","""Erectile dysfunction""","""PT"""
"""RXNORM:73494""","""meddra:10062237""","""telmisartan""","""Ingredient""","""Renal impairment""","""PT"""
"""RXNORM:276237""","""meddra:10062237""","""emtricitabine""","""Ingredient""","""Renal impairment""","""PT"""
